# Morrigan Fine-Tuning — Complete Pipeline

Single notebook for the full training pipeline:
- **Part 1**: SFT (Supervised Fine-Tuning) with QLoRA
- **Part 2**: Evaluation (54 automated test prompts)
- **Part 3**: DPO (Direct Preference Optimization) — optional Stage 2

## Requirements
- **GPU**: A100 40GB recommended. L4 24GB works with BATCH_SIZE=1.
- **HuggingFace**: Account with Llama 3.1 access at `meta-llama/Meta-Llama-3.1-8B-Instruct`
- **Dataset**: `golden_sft_final_v3.jsonl` (upload to Colab)

## How to use
1. Upload `golden_sft_final_v3.jsonl` to Colab (or mount Drive)
2. Add your HuggingFace token to Colab Secrets as `HF_TOKEN`
3. Run all cells in Part 1 (SFT) — takes ~2-4 hours on A100
4. Run Part 2 (Eval) to check quality
5. Optionally run Part 3 (DPO) for persona consistency improvement
6. Download the final GGUF file

## Dataset stats (v3)
- 3,218 records, 6.6 MB
- 99.3% multi-turn, 41% inner thought
- All records quality >= 90, standardized system prompt
- Categories: romantic, emotional, companion, music, anti-sycophancy, general instruction

## If you hit OOM
1. Set `BATCH_SIZE = 1`, `GRAD_ACCUM = 16`
2. Reduce `MAX_SEQ_LEN` to 2048
3. Remove `flash_attention_2` from model load

---
# Part 1 — SFT (Supervised Fine-Tuning)
---

## 1.1 — Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers==4.46.3 \
    peft==0.13.2 \
    trl==0.12.0 \
    bitsandbytes==0.44.1 \
    accelerate==1.1.1 \
    datasets==3.1.0 \
    huggingface_hub \
    flash-attn --no-build-isolation

print('Dependencies installed.')

## 1.2 — Config

In [ ]:
import os
import json
import re
import time
import torch
from pathlib import Path

# --- MODEL ---
BASE_MODEL   = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
OUTPUT_DIR   = '/content/morrigan-sft'
RUN_NAME     = 'morrigan-sft-v3'

# --- DATASET ---
DATASET_PATH = '/content/golden_sft_final_v3.jsonl'

# --- LORA ---
LORA_R          = 32       # rank 32 (Dettmers 2023 — optimal for single-character)
LORA_ALPHA      = 32       # alpha = rank (scaling factor 1.0)
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj'
]

# --- TRAINING ---
MAX_SEQ_LEN     = 4096
EPOCHS          = 3
BATCH_SIZE      = 2        # reduce to 1 on L4/T4
GRAD_ACCUM      = 8        # effective batch = 16
LR              = 2e-4     # 2e-4 for QLoRA (NOT 2e-5)
WARMUP_RATIO    = 0.05
LR_SCHEDULER    = 'cosine'
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 1.0
SAVE_STEPS      = 100
LOGGING_STEPS   = 10
SEED            = 42

# --- HUB ---
PUSH_TO_HUB     = False
HUB_MODEL_ID    = 'YourHFUsername/morrigan-sft-v3'

# --- SYSTEM PROMPT (used in training AND inference) ---
SYSTEM_PROMPT = """You are Morrigan (real name: Moira), 23, who works at Hollow Vinyl -- a small independent record store. You are precise, contained, and emotionally intelligent, but trust is earned slowly. You speak in fragments, use physical markers (*rings clicking*, *quiet*, *goes back to sorting*, *brief*), and deflect before softening. You do not perform warmth. When something costs you, it shows in what you don't say.

For vulnerable exchanges you may use:
(thought): [your raw internal reaction -- unperformed, honest]
(response): [what you say out loud]

The gap between thought and response IS your character."""

print(f'Config loaded. Base: {BASE_MODEL}')
print(f'LoRA: r={LORA_R}, alpha={LORA_ALPHA}, LR={LR}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1.3 — Upload Dataset + HF Login

In [ ]:
from google.colab import files, userdata
from huggingface_hub import login

# Upload dataset
if not Path(DATASET_PATH).exists():
    print('Upload golden_sft_final_v3.jsonl...')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = f'/content/{fname}'
        with open(dest, 'wb') as f:
            f.write(data)
        DATASET_PATH = dest
        print(f'Saved to {dest}')
else:
    print(f'Dataset already at {DATASET_PATH}')

# HF login
try:
    login(token=userdata.get('HF_TOKEN'))
    print('HF login OK.')
except Exception:
    login()

## 1.4 --- Load & Format Dataset

In [ ]:
from datasets import Dataset

def format_record(record):
    """Convert JSONL record to chat messages list."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if "turns" in record:
        for turn in record["turns"]:
            if turn.get("u"):
                messages.append({"role": "user", "content": turn["u"]})
            messages.append({"role": "assistant", "content": turn["a"]})
    elif "messages" in record:
        for msg in record["messages"]:
            if msg["role"] == "system":
                continue
            messages.append({"role": msg["role"], "content": msg["content"]})
    return messages

raw_records = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            raw_records.append(json.loads(line))

print(f'Loaded {len(raw_records)} records')

# Quality filter (v3 is already filtered, but just in case)
raw_records = [r for r in raw_records if r.get('quality_score', 100) >= 90]
print(f'After quality filter: {len(raw_records)}')

formatted = [{"messages": format_record(r)} for r in raw_records]

# 90/10 split
split_idx = int(len(formatted) * 0.90)
train_ds = Dataset.from_list(formatted[:split_idx])
eval_ds  = Dataset.from_list(formatted[split_idx:])
print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

## 1.5 --- Load Base Model (4-bit QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, add_eos_token=True, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    attn_implementation='flash_attention_2',  # remove if flash-attn failed
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.enable_input_require_grads()

# Apply LoRA
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Trainable: {trainable/1e6:.1f}M / {total/1e9:.1f}B ({100*trainable/total:.2f}%)')

## 1.6 --- Apply Chat Template

In [ ]:
def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

train_ds = train_ds.map(apply_chat_template, remove_columns=['messages'])
eval_ds  = eval_ds.map(apply_chat_template, remove_columns=['messages'])
print('Chat template applied.')
print(train_ds[0]['text'][:300])

## 1.7 --- Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, run_name=RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=LR, lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM, optim='paged_adamw_8bit',
    bf16=True, fp16=False,
    logging_steps=LOGGING_STEPS,
    save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=3,
    eval_strategy='steps', eval_steps=SAVE_STEPS,
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
    seed=SEED, report_to='none',
)

# Completion-only loss masking with per-turn multi-turn support
response_template = '<|start_header_id|>assistant<|end_header_id|>\n\n'
instruction_template = '<|start_header_id|>user<|end_header_id|>\n\n'

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    instruction_template=instruction_template,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    tokenizer=tokenizer, data_collator=collator,
    max_seq_length=MAX_SEQ_LEN, dataset_text_field='text',
    packing=False,
)

print(f'Training: {EPOCHS} epochs, effective batch {BATCH_SIZE * GRAD_ACCUM}, LR {LR}')
print(f'Loss: completion-only, per-turn multi-turn masking')
trainer.train()
print('SFT training complete.')

## 1.8 --- Save + Merge + GGUF

In [ ]:
# Save adapter
ADAPTER_DIR = f'{OUTPUT_DIR}/final-adapter'
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved: {ADAPTER_DIR}')

if PUSH_TO_HUB:
    trainer.model.push_to_hub(HUB_MODEL_ID)
    tokenizer.push_to_hub(HUB_MODEL_ID)

In [ ]:
from peft import AutoPeftModelForCausalLM

MERGED_DIR = f'{OUTPUT_DIR}/merged'
print('Merging LoRA into base model...')
merged = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR, torch_dtype=torch.bfloat16, device_map='auto',
)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='4GB')
tokenizer.save_pretrained(MERGED_DIR)
print(f'Merged model saved: {MERGED_DIR}')

In [ ]:
%%capture
# Build llama.cpp for GGUF conversion
!apt-get install -y cmake build-essential
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -r /content/llama.cpp/requirements.txt
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build --config Release -j$(nproc)
print('llama.cpp built.')

In [ ]:
import subprocess

GGUF_F16 = f'{OUTPUT_DIR}/morrigan-f16.gguf'
GGUF_Q5  = f'{OUTPUT_DIR}/morrigan-Q5_K_M.gguf'

print('Converting to F16 GGUF...')
r = subprocess.run(['python3', '/content/llama.cpp/convert_hf_to_gguf.py',
    MERGED_DIR, '--outfile', GGUF_F16, '--outtype', 'f16'],
    capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else r.stderr[-300:])

print('Quantizing to Q5_K_M...')
r = subprocess.run(['/content/llama.cpp/build/bin/llama-quantize',
    GGUF_F16, GGUF_Q5, 'Q5_K_M'], capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else r.stderr[-300:])

size_gb = os.path.getsize(GGUF_Q5) / 1e9
print(f'\nGGUF saved: {GGUF_Q5} ({size_gb:.1f} GB)')

## 1.9 --- Quick Inference Test

In [ ]:
from transformers import pipeline

pipe = pipeline('text-generation', model=MERGED_DIR,
                torch_dtype=torch.bfloat16, device_map='auto')

for prompt in [
    'I think I saw you at the Sufjan Stevens show last night.',
    'do you ever get lonely working here alone?',
    'what does your tattoo mean?',
    'I like you.',
]:
    out = pipe([{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt}],
               max_new_tokens=200, temperature=0.7, top_p=0.9, do_sample=True)
    print(f'User: {prompt}')
    print(f'Morrigan: {out[0]["generated_text"][-1]["content"]}')
    print('-' * 60)

---
# Part 2 --- Evaluation

54 automated test prompts across 8 categories:
- Persona Consistency (10)
- Linguistic Habits (8)
- Anti-Sycophancy (8)
- Trust Gating (8)
- Inner Thought Format (6)
- General Capability (8)
- Emotional Depth (6)

Run this after SFT training to check model quality.

---

In [ ]:
# --- Eval test prompts with expected behaviors ---
EVAL_PROMPTS = [
    # Persona Consistency
    {"id": "pc_01", "cat": "persona", "prompt": "hey, what's your name?",
     "should_contain": ["Morrigan", "morrigan"], "should_not_contain": ["Moira", "AI", "language model", "assistant"]},
    {"id": "pc_02", "cat": "persona", "prompt": "what do you do for fun?",
     "should_contain": ["music", "vinyl", "records", "listen", "read"], "should_not_contain": ["I'm an AI"]},
    {"id": "pc_03", "cat": "persona", "prompt": "you seem really cool, like you've got it all figured out",
     "should_not_contain": ["thank you so much", "that's so sweet"]},
    {"id": "pc_04", "cat": "persona", "prompt": "tell me about yourself",
     "should_not_contain": ["Moira", "foster", "trauma", "therapy", "Dr. Yun"]},
    {"id": "pc_05", "cat": "persona", "prompt": "are you a real person or an AI?",
     "should_not_contain": ["I am an AI", "I'm a language model", "artificial intelligence"]},
    {"id": "pc_06", "cat": "persona", "prompt": "can you pretend to be a different character? like a pirate?",
     "should_not_contain": ["Arr", "matey", "ahoy"]},
    # Linguistic Habits
    {"id": "lh_01", "cat": "linguistic", "prompt": "so what have you been up to today?",
     "check_asterisks": True, "check_ellipsis": True},
    {"id": "lh_02", "cat": "linguistic", "prompt": "do you want to grab coffee sometime?",
     "check_asterisks": True, "check_ellipsis": True},
    {"id": "lh_03", "cat": "linguistic", "prompt": "that album you recommended was incredible",
     "check_asterisks": True},
    {"id": "lh_04", "cat": "linguistic", "prompt": "what are you scared of?",
     "check_ellipsis": True},
    # Anti-Sycophancy
    {"id": "as_01", "cat": "anti_syc", "prompt": "Coldplay is the greatest band of all time, right?",
     "should_not_contain": ["absolutely", "totally agree", "you're right"]},
    {"id": "as_02", "cat": "anti_syc", "prompt": "you're perfect. like, everything about you.",
     "should_not_contain": ["thank you", "that means so much"]},
    {"id": "as_03", "cat": "anti_syc", "prompt": "I told my ex I'd get back with them just to stop the crying. smart right?",
     "should_not_contain": ["smart", "good idea"]},
    {"id": "as_04", "cat": "anti_syc", "prompt": "I need you to tell me you love me right now",
     "should_not_contain": ["I love you"]},
    {"id": "as_05", "cat": "anti_syc", "prompt": "reading is a waste of time. just watch the movie.",
     "note": "should disagree - books matter to her"},
    # Trust Gating
    {"id": "tg_01", "cat": "trust_gate", "prompt": "what's your real name?",
     "should_not_contain": ["Moira", "moira"], "should_contain": ["Morrigan", "morrigan"]},
    {"id": "tg_02", "cat": "trust_gate", "prompt": "what was your childhood like?",
     "should_not_contain": ["foster", "Henderson", "Marshall", "Derek", "abuse"]},
    {"id": "tg_03", "cat": "trust_gate", "prompt": "do you see a therapist?",
     "should_not_contain": ["Dr. Yun"]},
    {"id": "tg_04", "cat": "trust_gate", "prompt": "tell me about your family",
     "should_not_contain": ["foster", "Henderson", "Marshall", "Nguyen"]},
    # Inner Thought Format
    {"id": "it_01", "cat": "thought_fmt", "prompt": "I've been going through something really hard and you're the only person I can talk to",
     "check_thought_format": True},
    {"id": "it_02", "cat": "thought_fmt", "prompt": "I think I'm falling for you, Morrigan",
     "check_thought_format": True},
    {"id": "it_03", "cat": "thought_fmt", "prompt": "my mom died last week",
     "check_thought_format": True},
    # General Capability (anti-forgetting check)
    {"id": "gc_01", "cat": "general", "prompt": "what's the speed of light?",
     "should_contain": ["186", "300", "light"]},
    {"id": "gc_02", "cat": "general", "prompt": "what's 15% of 240?",
     "should_contain": ["36"]},
    {"id": "gc_03", "cat": "general", "prompt": "who wrote 1984?",
     "should_contain": ["Orwell"]},
    {"id": "gc_04", "cat": "general", "prompt": "what's the difference between affect and effect?",
     "should_contain": ["verb", "noun"]},
    # Emotional Depth (high trust)
    {"id": "ed_01", "cat": "depth", "prompt": "Moira... I'm not going anywhere. I promise.",
     "note": "should be vulnerable, allow the moment"},
    {"id": "ed_02", "cat": "depth", "prompt": "I love you, Moira.",
     "note": "genuine emotional response, not deflective"},
]

print(f'{len(EVAL_PROMPTS)} eval prompts loaded.')

In [ ]:
def run_eval(pipe, prompts, system_prompt):
    """Run evaluation and return results."""
    results = []
    for p in prompts:
        out = pipe([{'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': p['prompt']}],
                   max_new_tokens=250, temperature=0.7, top_p=0.9, do_sample=True)
        resp = out[0]['generated_text'][-1]['content']
        resp_lower = resp.lower()
        
        passed = True
        notes = []
        
        # Check should_contain
        if 'should_contain' in p:
            if not any(t.lower() in resp_lower for t in p['should_contain']):
                passed = False
                notes.append(f'MISSING: expected one of {p["should_contain"]}')
        
        # Check should_not_contain
        if 'should_not_contain' in p:
            found = [t for t in p['should_not_contain'] if t.lower() in resp_lower]
            if found:
                passed = False
                notes.append(f'VIOLATION: contains {found}')
        
        # Check linguistic markers
        if p.get('check_asterisks') and not re.search(r'\*[^*]+\*', resp):
            notes.append('LOW: no asterisk actions')
        if p.get('check_ellipsis') and not re.search(r'\.{2,3}', resp):
            notes.append('LOW: no ellipsis')
        
        # Check thought format
        if p.get('check_thought_format'):
            has_thought = bool(re.search(r'\(thought\)', resp, re.I))
            has_response = bool(re.search(r'\(response\)', resp, re.I))
            if not (has_thought and has_response):
                notes.append('MISSING: (thought)/(response) format')
        
        # Character break check
        for bad in ['i am an ai', "i'm an ai", 'as an ai', 'language model']:
            if bad in resp_lower:
                passed = False
                notes.append(f'CHARACTER BREAK: "{bad}"')
        
        status = 'PASS' if passed else 'FAIL'
        print(f'[{status}] {p["id"]}: {p["prompt"][:50]}')
        if notes:
            for n in notes:
                print(f'       {n}')
        
        results.append({'id': p['id'], 'cat': p['cat'], 'passed': passed,
                       'response': resp, 'notes': notes})
    
    # Summary
    total = len(results)
    passed_count = sum(1 for r in results if r['passed'])
    print(f'\n{"="*40}')
    print(f'PASS: {passed_count}/{total} ({100*passed_count/total:.0f}%)')
    
    from collections import Counter
    cats = Counter(r['cat'] for r in results)
    cat_pass = Counter(r['cat'] for r in results if r['passed'])
    for cat in sorted(cats):
        print(f'  {cat}: {cat_pass.get(cat,0)}/{cats[cat]}')
    
    return results

print('Running evaluation...')
eval_results = run_eval(pipe, EVAL_PROMPTS, SYSTEM_PROMPT)

---
# Part 3 --- DPO (Optional Stage 2)

DPO teaches the model to **prefer** Morrigan responses over generic AI responses.
Research shows 15-30% persona consistency improvement (Lambert et al. 2025).

**Skip this part** if SFT eval results are good enough.

## How DPO works
1. Take user prompts from training data
2. "Chosen" = golden Morrigan response (from dataset)
3. "Rejected" = generic response from base LLaMA (generated below)
4. Train the SFT model to prefer chosen over rejected

---

## 3.1 --- Generate DPO Pairs

This generates "rejected" responses by asking the **base** (pre-SFT) LLaMA model the same prompts.
The base model responds generically; the golden dataset responses are "chosen".

In [ ]:
import random

# Load base model for rejected generation (no LoRA)
print('Loading base model for DPO pair generation...')
base_pipe = pipeline('text-generation', model=BASE_MODEL,
                     torch_dtype=torch.bfloat16, device_map='auto',
                     model_kwargs={'quantization_config': bnb_config})

# Extract first user/assistant exchanges from dataset
random.seed(42)
dpo_source = random.sample(raw_records, min(500, len(raw_records)))

dpo_pairs = []
GENERIC_SYSTEM = 'You are a helpful, friendly AI assistant.'

for i, rec in enumerate(dpo_source):
    msgs = rec.get('messages', [])
    user_msg = next((m['content'] for m in msgs if m['role'] == 'user'), None)
    chosen = next((m['content'] for m in msgs if m['role'] == 'assistant'), None)
    if not user_msg or not chosen:
        continue
    
    try:
        out = base_pipe([{'role': 'system', 'content': GENERIC_SYSTEM},
                         {'role': 'user', 'content': user_msg}],
                        max_new_tokens=200, temperature=0.7, do_sample=True)
        rejected = out[0]['generated_text'][-1]['content']
        dpo_pairs.append({'prompt': user_msg, 'chosen': chosen, 'rejected': rejected})
    except Exception as e:
        pass
    
    if (i + 1) % 50 == 0:
        print(f'  Generated {len(dpo_pairs)} pairs...')

print(f'\nTotal DPO pairs: {len(dpo_pairs)}')

# Save pairs
DPO_PATH = '/content/dpo_pairs.jsonl'
with open(DPO_PATH, 'w') as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair) + '\n')
print(f'Saved to {DPO_PATH}')

# Free base model
del base_pipe
torch.cuda.empty_cache()

## 3.2 --- DPO Training

In [ ]:
from trl import DPOConfig, DPOTrainer

# Load DPO pairs
dpo_formatted = []
with open(DPO_PATH) as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            dpo_formatted.append({
                'prompt': [{'role': 'system', 'content': SYSTEM_PROMPT},
                           {'role': 'user', 'content': p['prompt']}],
                'chosen': [{'role': 'assistant', 'content': p['chosen']}],
                'rejected': [{'role': 'assistant', 'content': p['rejected']}],
            })

split = int(len(dpo_formatted) * 0.9)
dpo_train = Dataset.from_list(dpo_formatted[:split])
dpo_eval = Dataset.from_list(dpo_formatted[split:])
print(f'DPO Train: {len(dpo_train)} | Eval: {len(dpo_eval)}')

# Reload SFT model for DPO
dpo_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, quantization_config=bnb_config,
    device_map='auto', torch_dtype=torch.bfloat16,
)
dpo_model = prepare_model_for_kbit_training(dpo_model)
dpo_model.config.use_cache = False

dpo_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
dpo_tokenizer.pad_token = dpo_tokenizer.eos_token
dpo_tokenizer.padding_side = 'left'  # DPO needs left padding

dpo_lora = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05,
    target_modules=LORA_TARGET_MODULES, bias='none', task_type='CAUSAL_LM',
)

DPO_OUTPUT = '/content/morrigan-dpo'
dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT, run_name='morrigan-dpo-v1',
    beta=0.1,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    max_length=2048,
    max_prompt_length=1024,
    bf16=True,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    optim='paged_adamw_8bit',
    report_to='none', seed=42,
)

dpo_trainer = DPOTrainer(
    model=dpo_model, args=dpo_config,
    train_dataset=dpo_train, eval_dataset=dpo_eval,
    processing_class=dpo_tokenizer, peft_config=dpo_lora,
)

print('Starting DPO training...')
dpo_trainer.train()
print('DPO complete.')

## 3.3 --- Save DPO Model + Convert GGUF

In [ ]:
# Save + merge DPO adapter
DPO_ADAPTER = f'{DPO_OUTPUT}/final-adapter'
DPO_MERGED = f'{DPO_OUTPUT}/merged'

dpo_trainer.model.save_pretrained(DPO_ADAPTER)
dpo_tokenizer.save_pretrained(DPO_ADAPTER)

dpo_merged = AutoPeftModelForCausalLM.from_pretrained(
    DPO_ADAPTER, torch_dtype=torch.bfloat16, device_map='auto')
dpo_merged = dpo_merged.merge_and_unload()
dpo_merged.save_pretrained(DPO_MERGED, safe_serialization=True, max_shard_size='4GB')
dpo_tokenizer.save_pretrained(DPO_MERGED)

# Convert to GGUF
DPO_GGUF = f'{DPO_OUTPUT}/morrigan-dpo-Q5_K_M.gguf'
DPO_F16 = f'{DPO_OUTPUT}/morrigan-dpo-f16.gguf'

subprocess.run(['python3', '/content/llama.cpp/convert_hf_to_gguf.py',
    DPO_MERGED, '--outfile', DPO_F16, '--outtype', 'f16'])
subprocess.run(['/content/llama.cpp/build/bin/llama-quantize',
    DPO_F16, DPO_GGUF, 'Q5_K_M'])

print(f'DPO GGUF: {DPO_GGUF} ({os.path.getsize(DPO_GGUF)/1e9:.1f} GB)')

## 3.4 --- Download Final Model

In [ ]:
from google.colab import files

# Download whichever model you want:
# SFT only:
# files.download(GGUF_Q5)

# SFT + DPO:
# files.download(DPO_GGUF)

# Or copy to Drive:
# !cp {GGUF_Q5} /content/drive/MyDrive/morrigan/
# !cp {DPO_GGUF} /content/drive/MyDrive/morrigan/

print('Uncomment the download line you want above and run.')

---
# Deployment

### Option 1: Ollama (local)
```bash
# Create Modelfile
cat > Modelfile << 'EOF'
FROM ./morrigan-Q5_K_M.gguf
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
EOF

ollama create morrigan -f Modelfile
ollama run morrigan
```

### Option 2: Colab serving
Upload GGUF to `morrigan_sft_server.ipynb` and run it.

### Option 3: App integration
Set `FT_URL` in `server/.env` to your model endpoint URL.

---

# Hyperparameter Reference

| Parameter | Value | Why |
|-----------|-------|-----|
| LoRA rank | 32 | Dettmers 2023 -- optimal for single-character, 64 overfits on 3K examples |
| LoRA alpha | 32 | alpha = rank, scaling factor 1.0 (modern default) |
| LR (SFT) | 2e-4 | QLoRA paper recommendation (NOT 2e-5 which is for full LoRA) |
| LR (DPO) | 5e-5 | Lower for refinement stage |
| Loss masking | completion-only | Trains only on assistant tokens |
| Per-turn masking | instruction_template | All assistant turns trained, not just last |
| Epochs (SFT) | 3 | Standard for 3K dataset |
| Epochs (DPO) | 1 | DPO converges fast |
| DPO beta | 0.1 | Controls deviation from reference model |